In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "database").is_dir() and (p / "code").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Abra o notebook em Codigos ou em uma de suas subpastas.")
DATABASE_DIR = PROJECT_ROOT / "database"
RESULT_DIR = PROJECT_ROOT / "result" / "feedback_imediato"
RESULT_DIR.mkdir(parents=True, exist_ok=True)
# Cada experimento mantém seus próprios dataframes derivados.
PROCESSED_DIR = RESULT_DIR / "processados"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(PROJECT_ROOT / "code" / "visualizacao"))


In [2]:
import re
import numpy as np
import pandas as pd

from sklearn.linear_model import (
    Perceptron as SklearnPerceptron,
    PassiveAggressiveClassifier as SklearnPassiveAggressiveClassifier,
    SGDClassifier as SklearnSGDClassifier
)
from sklearn.naive_bayes import (
    MultinomialNB as SklearnMultinomialNB
)
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix
)


# ============================================================
# 1. CARREGA A BASE FAKERECOGNA
# ============================================================

CAMINHO_FAKERECOGNA = str(DATABASE_DIR / "FakeRecogna" / "FakeRecogna.xlsx")

df_recogna = pd.read_excel(
    CAMINHO_FAKERECOGNA
)


# ============================================================
# 2. TRATAMENTO DAS DATAS
# ============================================================

def converter_data_fakerecogna(valor):
    """
    Converte os diferentes formatos de data encontrados
    na base FakeRecogna.
    """

    if pd.isna(valor):
        return pd.NaT

    texto = str(valor).strip()

    # Formatos como:
    # 27/03/2020
    # 27/03/2020 18h25
    # Publicado em 20/05/21 04:30
    padrao_br = re.search(
        r"(\d{1,2})/(\d{1,2})/(\d{2,4})",
        texto
    )

    if padrao_br:
        dia, mes, ano = map(
            int,
            padrao_br.groups()
        )

        if ano < 100:
            ano += 2000

        try:
            return pd.Timestamp(
                year=ano,
                month=mes,
                day=dia
            )
        except ValueError:
            return pd.NaT


    # Formatos como:
    # 2020-03-31
    # 2020/11/20
    padrao_ano_primeiro = re.search(
        r"(\d{4})[-/](\d{1,2})[-/](\d{1,2})",
        texto
    )

    if padrao_ano_primeiro:
        ano, mes, dia = map(
            int,
            padrao_ano_primeiro.groups()
        )

        try:
            return pd.Timestamp(
                year=ano,
                month=mes,
                day=dia
            )
        except ValueError:
            return pd.NaT

    return pd.NaT


df_recogna["data_publicacao"] = (
    df_recogna["Data"]
    .apply(converter_data_fakerecogna)
)


# ============================================================
# 3. PREPARA O TEXTO
# ============================================================

# Junta título, subtítulo e notícia.
# Isso nos dá mais informação textual para o classificador.

df_recogna["texto_modelo"] = (
    df_recogna[
        ["Titulo", "Subtitulo", "Noticia"]
    ]
    .fillna("")
    .astype(str)
    .agg(" ".join, axis=1)
    .str.strip()
)


# ============================================================
# 4. CORRIGE OS RÓTULOS
# ============================================================

# Fake.br:
# fake = 1
# true = 0
#
# FakeRecogna:
# fake = 0
# true = 1
#
# Portanto, precisamos inverter.

df_recogna["rotulo"] = (
    1 - pd.to_numeric(
        df_recogna["Classe"],
        errors="coerce"
    )
)


df_recogna = df_recogna.dropna(
    subset=[
        "data_publicacao",
        "texto_modelo",
        "rotulo"
    ]
).copy()

df_recogna["rotulo"] = (
    df_recogna["rotulo"]
    .astype(int)
)


# ============================================================
# 5. MODELOS INICIAIS
# ============================================================

# Cada modelo possui uma variável própria.
# Isso permite testar e treinar os quatro modelos
# independentemente, sem sobrescrever os demais.

Perceptron_offline = SklearnPerceptron(
    random_state=42
)

PassiveAggressive_offline = (
    SklearnPassiveAggressiveClassifier(
        random_state=42
    )
)

MultinomialNB_offline = (
    SklearnMultinomialNB()
)

SGD_offline = SklearnSGDClassifier(
    random_state=42
)


# Treinamento inicial com a mesma Fake.br
# utilizada anteriormente.

Perceptron_offline.fit(
    X_inicial,
    y_inicial
)

PassiveAggressive_offline.fit(
    X_inicial,
    y_inicial
)

MultinomialNB_offline.fit(
    X_inicial,
    y_inicial
)

SGD_offline.fit(
    X_inicial,
    y_inicial
)


# ============================================================
# 6. FUNÇÃO AUXILIAR PARA ESCOLHER O MÊS
# ============================================================

INICIO_POS_CORTE = pd.Timestamp(
    "2020-01-01"
)


def selecionar_mes_fakerecogna(
    mes_apos_corte
):
    if mes_apos_corte < 1:
        raise ValueError(
            "mes_apos_corte deve ser >= 1"
        )

    inicio = (
        INICIO_POS_CORTE
        + pd.DateOffset(
            months=mes_apos_corte - 1
        )
    )

    fim = (
        inicio
        + pd.DateOffset(months=1)
    )

    dados_mes = df_recogna[
        (
            df_recogna["data_publicacao"]
            >= inicio
        )
        &
        (
            df_recogna["data_publicacao"]
            < fim
        )
    ].copy()

    return dados_mes, inicio, fim


# ============================================================
# 7. CLASSE DE TESTE E MÉTRICAS
# ============================================================

class AvaliacaoFakeRecogna:
    """
    Interface de teste para um modelo específico.

    Exemplos:
        SGDClassifier.recall(2)
        SGDClassifier.f1macro(2)
        SGDClassifier.matriz_confusao(2)
    """

    def __init__(
        self,
        modelo,
        nome_modelo
    ):
        self.modelo = modelo
        self.nome_modelo = nome_modelo

    def _avaliar(
        self,
        mes_apos_corte
    ):
        dados_mes, inicio, fim = (
            selecionar_mes_fakerecogna(
                mes_apos_corte
            )
        )

        if dados_mes.empty:
            return None

        X_teste = vetorizador.transform(
            dados_mes["texto_modelo"]
        )

        y_teste = (
            dados_mes["rotulo"]
            .to_numpy()
        )

        predicoes = self.modelo.predict(
            X_teste
        )

        resultado = dados_mes[
            [
                "Titulo",
                "data_publicacao",
                "rotulo"
            ]
        ].copy()

        resultado["predicao"] = predicoes

        return {
            "inicio": inicio,
            "fim": fim,
            "numero_noticias": len(dados_mes),
            "fake": int(
                (y_teste == 1).sum()
            ),
            "true": int(
                (y_teste == 0).sum()
            ),
            "accuracy": accuracy_score(
                y_teste,
                predicoes
            ),
            "f1macro": f1_score(
                y_teste,
                predicoes,
                average="macro",
                zero_division=0
            ),
            "precision": precision_score(
                y_teste,
                predicoes,
                average="macro",
                zero_division=0
            ),
            "recall": recall_score(
                y_teste,
                predicoes,
                average="macro",
                zero_division=0
            ),
            "matriz_confusao": confusion_matrix(
                y_teste,
                predicoes,
                labels=[0, 1]
            ),
            "resultado": resultado
        }

    def _obter(
        self,
        mes_apos_corte,
        chave
    ):
        avaliacao = self._avaliar(
            mes_apos_corte
        )

        if avaliacao is None:
            return None

        return avaliacao[chave]

    def numero_noticias(
        self,
        mes_apos_corte
    ):
        return self._obter(
            mes_apos_corte,
            "numero_noticias"
        )

    def fake(
        self,
        mes_apos_corte
    ):
        return self._obter(
            mes_apos_corte,
            "fake"
        )

    def true(
        self,
        mes_apos_corte
    ):
        return self._obter(
            mes_apos_corte,
            "true"
        )

    def accuracy(
        self,
        mes_apos_corte
    ):
        return self._obter(
            mes_apos_corte,
            "accuracy"
        )

    def f1macro(
        self,
        mes_apos_corte
    ):
        return self._obter(
            mes_apos_corte,
            "f1macro"
        )

    def precision(
        self,
        mes_apos_corte
    ):
        return self._obter(
            mes_apos_corte,
            "precision"
        )

    def recall(
        self,
        mes_apos_corte
    ):
        return self._obter(
            mes_apos_corte,
            "recall"
        )

    def matriz_confusao(
        self,
        mes_apos_corte
    ):
        return self._obter(
            mes_apos_corte,
            "matriz_confusao"
        )

    def testar(
        self,
        mes_apos_corte
    ):
        avaliacao = self._avaliar(
            mes_apos_corte
        )

        if avaliacao is None:
            dados_mes, inicio, fim = (
                selecionar_mes_fakerecogna(
                    mes_apos_corte
                )
            )

            print(
                f"Não existem notícias para "
                f"{inicio.strftime('%m/%Y')}."
            )
            return None

        print(
            "========================================"
        )

        print(
            f"TESTE — {self.nome_modelo} — "
            f"{avaliacao['inicio'].strftime('%m/%Y')}"
        )

        print(
            "========================================"
        )

        print(
            f"Número de notícias: "
            f"{avaliacao['numero_noticias']}"
        )

        print(
            f"Fake: {avaliacao['fake']}"
        )

        print(
            f"True: {avaliacao['true']}"
        )

        print()

        print(
            f"Accuracy:  "
            f"{avaliacao['accuracy']:.4f}"
        )

        print(
            f"F1 Macro:  "
            f"{avaliacao['f1macro']:.4f}"
        )

        print(
            f"Precision: "
            f"{avaliacao['precision']:.4f}"
        )

        print(
            f"Recall:    "
            f"{avaliacao['recall']:.4f}"
        )

        print()

        print(
            "Matriz de confusão:"
        )

        print(
            avaliacao["matriz_confusao"]
        )

        return avaliacao["resultado"]

    def treinar(
        self,
        mes_apos_corte
    ):
        return treinar_mes_fakerecogna(
            self.modelo,
            mes_apos_corte
        )


# ============================================================
# 8. FUNÇÃO DE TESTE COM MODELO COMO INPUT
# ============================================================

def testar_mes_fakerecogna(
    modelo,
    mes_apos_corte
):
    avaliador = AvaliacaoFakeRecogna(
        modelo=modelo,
        nome_modelo=type(modelo).__name__
    )

    return avaliador.testar(
        mes_apos_corte
    )


# ============================================================
# 9. FUNÇÃO DE TREINO COM MODELO COMO INPUT
# ============================================================

def treinar_mes_fakerecogna(
    modelo,
    mes_apos_corte
):
    dados_mes, inicio, fim = (
        selecionar_mes_fakerecogna(
            mes_apos_corte
        )
    )

    if dados_mes.empty:
        print(
            f"Não existem notícias para "
            f"{inicio.strftime('%m/%Y')}."
        )
        return modelo

    X_treino_mes = vetorizador.transform(
        dados_mes["texto_modelo"]
    )

    y_treino_mes = (
        dados_mes["rotulo"]
        .to_numpy()
    )

    # Os quatro modelos escolhidos suportam
    # atualização incremental via partial_fit.
    modelo.partial_fit(
        X_treino_mes,
        y_treino_mes,
        classes=np.array([0, 1])
    )



    return modelo


# ============================================================
# 10. INTERFACES POR MODELO
# ============================================================

# Estas quatro interfaces deixam cada resultado separado.
#
# Exemplos:
# SGDClassifier.recall(2)
# SGDClassifier.f1macro(2)
# MultinomialNB.accuracy(1)
# Perceptron.matriz_confusao(3)

Perceptron = AvaliacaoFakeRecogna(
    Perceptron_offline,
    "Perceptron"
)

PassiveAggressiveClassifier = (
    AvaliacaoFakeRecogna(
        PassiveAggressive_offline,
        "PassiveAggressiveClassifier"
    )
)

MultinomialNB = AvaliacaoFakeRecogna(
    MultinomialNB_offline,
    "MultinomialNB"
)

SGDClassifier = AvaliacaoFakeRecogna(
    SGD_offline,
    "SGDClassifier"
)


/home/joao-lucas-nogueira/Área de trabalho/Projetos/IC/Codigos/.venv/lib/python3.14/site-packages/sklearn/utils/deprecation.py:71: FutureWarning: Class PassiveAggressiveClassifier is deprecated; this is deprecated in version 1.8 and will be removed in 1.10. Use `SGDClassifier(loss='hinge', penalty=None, learning_rate='pa1', eta0=1.0)` instead.
  warnings.warn(msg, category=FutureWarning)


NameError: name 'X_inicial' is not defined

# Fake.Br — treinamento com divisão temporal

Notebook iniciado em 05/08/2026.

Neste experimento:

- **Treino:** notícias publicadas até **31/12/2019**, inclusive.
- **Teste:** notícias publicadas após **31/12/2019**, isto é, a partir de **01/01/2020**.
- **Avaliação online:** os dados pós-corte são processados em ordem cronológica, com previsão antes da atualização do modelo.


In [ ]:
from pathlib import Path
import pandas as pd


In [ ]:
CAMINHO_BASE = DATABASE_DIR / "Fake.br" / "corpus" / "full_texts"

COLUNAS_METADADOS = [
    "autor",
    "link",
    "categoria",
    "data_publicacao",
    "numero_tokens",
    "numero_palavras_sem_pontuacao",
    "numero_types",
    "numero_links",
    "numero_maiusculas",
    "numero_verbos",
    "numero_verbos_subjuntivo_imperativo",
    "numero_substantivos",
    "numero_adjetivos",
    "numero_adverbios",
    "numero_verbos_modais",
    "numero_pronomes_singular_primeira_segunda_pessoa",
    "numero_pronomes_plural_primeira_pessoa",
    "numero_pronomes",
    "pausalidade",
    "numero_caracteres",
    "tamanho_medio_sentenca",
    "tamanho_medio_palavra",
    "percentual_erros_ortograficos",
    "emotividade",
    "diversidade"
]

COLUNAS_NUMERICAS = COLUNAS_METADADOS[4:]


def ler_arquivo(caminho):
    try:
        return caminho.read_text(encoding="utf-8")
    except UnicodeDecodeError:
        return caminho.read_text(encoding="latin-1")

In [ ]:
def carregar_classe(caminho_base, classe):
    """
    Carrega todos os textos e metadados de uma classe.

    classe deve ser:
    - "fake"
    - "true"
    """

    pasta_textos = caminho_base / classe
    pasta_metadados = caminho_base / f"{classe}-meta-information"

    registros = []
    problemas = []

    arquivos_texto = sorted(
        pasta_textos.glob("*.txt"),
        key=lambda arquivo: int(arquivo.stem)
    )

    for caminho_texto in arquivos_texto:
        identificador = int(caminho_texto.stem)

        caminho_metadados = (
            pasta_metadados / f"{identificador}-meta.txt"
        )

        if not caminho_metadados.exists():
            problemas.append(
                f"Metadados não encontrados para o arquivo {identificador}"
            )
            continue

        texto_noticia = ler_arquivo(caminho_texto).strip()

        linhas_metadados = [
            linha.strip()
            for linha in ler_arquivo(caminho_metadados).splitlines()
        ]

        # Remove somente linhas vazias no final do arquivo.
        while linhas_metadados and linhas_metadados[-1] == "":
            linhas_metadados.pop()

        if len(linhas_metadados) != len(COLUNAS_METADADOS):
            problemas.append(
                f"Arquivo {caminho_metadados.name}: "
                f"esperadas {len(COLUNAS_METADADOS)} linhas, "
                f"encontradas {len(linhas_metadados)}"
            )
            continue

        metadados = dict(
            zip(COLUNAS_METADADOS, linhas_metadados)
        )

        registro = {
            "id": identificador,
            "texto": texto_noticia,
            "classe": classe,

            # Convenção adotada:
            # fake = 1
            # true = 0
            "rotulo": 1 if classe == "fake" else 0,

            **metadados
        }

        registros.append(registro)

    if problemas:
        print(f"Foram encontrados {len(problemas)} problemas.")

        for problema in problemas[:10]:
            print(problema)

    return pd.DataFrame(registros)




In [ ]:
df_fake = carregar_classe(CAMINHO_BASE, "fake")
df_true = carregar_classe(CAMINHO_BASE, "true")

df = pd.concat(
    [df_fake, df_true],
    ignore_index=True
)

In [ ]:
# Preserva o valor original para facilitar o diagnóstico de datas inválidas.
df["data_publicacao_original"] = df["data_publicacao"].copy()

# Converte as datas no formato dia/mês/ano.
df["data_publicacao"] = pd.to_datetime(
    df["data_publicacao_original"].astype("string").str.strip(),
    format="%d/%m/%Y",
    errors="coerce"
)

# Converte os atributos numéricos dos metadados.
df[COLUNAS_NUMERICAS] = df[COLUNAS_NUMERICAS].apply(
    pd.to_numeric,
    errors="coerce"
)


In [ ]:
print(df.shape)
print(df.columns)


In [ ]:
df.head()

In [ ]:
df["classe"].value_counts()

In [ ]:
# ============================================================
# VALIDAÇÃO E COBERTURA TEMPORAL DA BASE
# ============================================================

print("Tipo da coluna:", df["data_publicacao"].dtype)
print("Datas inválidas ou ausentes:", df["data_publicacao"].isna().sum())
print("Data mais antiga:", df["data_publicacao"].min())
print("Data mais recente:", df["data_publicacao"].max())

print("\nQuantidade de notícias por ano:")
quantidade_por_ano = (
    df.dropna(subset=["data_publicacao"])
      .groupby(df["data_publicacao"].dt.year)
      .size()
      .sort_index()
)
print(quantidade_por_ano)

# Exibe exemplos de datas que não puderam ser convertidas, caso existam.
datas_invalidas = df.loc[
    df["data_publicacao"].isna(),
    ["id", "classe", "data_publicacao_original"]
]

if not datas_invalidas.empty:
    print("\nExemplos de datas inválidas ou ausentes:")
    display(datas_invalidas.head(20))


In [ ]:
# ============================================================
# SALVAMENTO OPCIONAL DO DATAFRAME PROCESSADO
# ============================================================

# Pickle preserva os tipos das colunas, inclusive datetime.
df.to_pickle(str(PROCESSED_DIR / "fakebr_dataframe.pkl"))

# CSV facilita a inspeção em outros programas.
df.to_csv(
    str(PROCESSED_DIR / "fakebr_dataframe.csv"),
    index=False,
    encoding="utf-8",
    date_format="%Y-%m-%d"
)


In [ ]:
import numpy as np

from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.linear_model import SGDClassifier


# ============================================================
# 1. BASE INICIAL: TODA A FAKE.BR
# ============================================================

# Remove registros sem texto ou rótulo
dados_iniciais = df.dropna(
    subset=["texto", "rotulo"]
).copy()

# Garante que os rótulos sejam números inteiros
dados_iniciais["rotulo"] = dados_iniciais["rotulo"].astype(int)


# ============================================================
# 2. VETORIZADOR PARA APRENDIZADO ONLINE
# ============================================================

# O HashingVectorizer não precisa aprender um vocabulário fixo.
# Por isso, ele pode transformar continuamente novos textos.
vetorizador = HashingVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    n_features=2**18,
    alternate_sign=False,
    norm="l2"
)

# Transforma toda a Fake.br em vetores numéricos
X_inicial = vetorizador.transform(
    dados_iniciais["texto"]
)

y_inicial = dados_iniciais["rotulo"].to_numpy()




In [ ]:
from sklearn.model_selection import cross_val_predict
from sklearn.linear_model import (
    Perceptron as SklearnPerceptron,
    PassiveAggressiveClassifier as SklearnPassiveAggressiveClassifier,
    SGDClassifier as SklearnSGDClassifier
)
from sklearn.naive_bayes import MultinomialNB as SklearnMultinomialNB


cvC = 10

classifier = SklearnPerceptron(random_state=42)
pred = cross_val_predict(
    classifier,
    X_inicial,
    y_inicial,
    cv=cvC,
    verbose=True,
    n_jobs=2
)

classifier2 = SklearnPassiveAggressiveClassifier(random_state=42)
pred2 = cross_val_predict(
    classifier2,
    X_inicial,
    y_inicial,
    cv=cvC,
    verbose=True,
    n_jobs=2
)

classifier3 = SklearnMultinomialNB()
pred3 = cross_val_predict(
    classifier3,
    X_inicial,
    y_inicial,
    cv=cvC,
    verbose=True,
    n_jobs=2
)

classifier4 = SklearnSGDClassifier(random_state=42)
pred4 = cross_val_predict(
    classifier4,
    X_inicial,
    y_inicial,
    cv=cvC,
    verbose=True,
    n_jobs=2
)


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score
)

score = accuracy_score(y_inicial, pred)
score2 = accuracy_score(y_inicial, pred2)
score3 = accuracy_score(y_inicial, pred3)
score4 = accuracy_score(y_inicial, pred4)


In [ ]:
ps1 = precision_score(
    y_inicial,
    pred,
    average="macro",
    zero_division=0
)
r1 = recall_score(
    y_inicial,
    pred,
    average="macro",
    zero_division=0
)

ps2 = precision_score(
    y_inicial,
    pred2,
    average="macro",
    zero_division=0
)
r2 = recall_score(
    y_inicial,
    pred2,
    average="macro",
    zero_division=0
)

ps3 = precision_score(
    y_inicial,
    pred3,
    average="macro",
    zero_division=0
)
r3 = recall_score(
    y_inicial,
    pred3,
    average="macro",
    zero_division=0
)

ps4 = precision_score(
    y_inicial,
    pred4,
    average="macro",
    zero_division=0
)
r4 = recall_score(
    y_inicial,
    pred4,
    average="macro",
    zero_division=0
)


In [ ]:
y_train2 = y_inicial
X_train = X_inicial


In [ ]:
f1_geral = f1_score(
    y_train2,
    pred,
    average="macro"
)
f1_geral2 = f1_score(
    y_train2,
    pred2,
    average="macro"
)
f1_geral3 = f1_score(
    y_train2,
    pred3,
    average="macro"
)
f1_geral4 = f1_score(
    y_train2,
    pred4,
    average="macro"
)

df1_geral = f1_score(
    y_train2,
    pred,
    average=None
)
df1_gera2 = f1_score(
    y_train2,
    pred2,
    average=None
)
df1_gera3 = f1_score(
    y_train2,
    pred3,
    average=None
)
df1_gera4 = f1_score(
    y_train2,
    pred4,
    average=None
)


In [ ]:
print(
    "Perceptron - Accuracy:",
    score,
    ", F1_Score:",
    f1_geral,
    " Precision:",
    ps1,
    " Recall:",
    r1,
    "\n F1-Classe",
    df1_geral
)

print(
    "PassiveAggressiveClassifier - Accuracy:",
    score2,
    ", F1_Score:",
    f1_geral2,
    " Precision:",
    ps2,
    " Recall:",
    r2,
    "\n F1-Classe",
    df1_gera2
)

print(
    "MultinomialNB - Accuracy:",
    score3,
    ", F1_Score:",
    f1_geral3,
    " Precision:",
    ps3,
    " Recall:",
    r3,
    "\n F1-Classe",
    df1_gera3
)

print(
    "SGDClassifier - Accuracy:",
    score4,
    ", F1_Score:",
    f1_geral4,
    " Precision:",
    ps4,
    " Recall:",
    r4,
    "\n F1-Classe",
    df1_gera4
)


In [ ]:
from imblearn.over_sampling import SMOTE

sm = SMOTE(random_state=42)

X_res, y_res = sm.fit_resample(
    X_train,
    y_train2
)

X_train = X_res
y_train2 = y_res


from sklearn.model_selection import cross_val_predict

cvC = 10

classifier = SklearnPerceptron(random_state=42)
pred = cross_val_predict(
    classifier,
    X_train,
    y_train2,
    cv=cvC,
    verbose=True,
    n_jobs=2
)

classifier2 = SklearnPassiveAggressiveClassifier(random_state=42)
pred2 = cross_val_predict(
    classifier2,
    X_train,
    y_train2,
    cv=cvC,
    verbose=True,
    n_jobs=2
)

classifier3 = SklearnMultinomialNB()
pred3 = cross_val_predict(
    classifier3,
    X_train,
    y_train2,
    cv=cvC,
    verbose=True,
    n_jobs=2
)

classifier4 = SklearnSGDClassifier(random_state=42)
pred4 = cross_val_predict(
    classifier4,
    X_train,
    y_train2,
    cv=cvC,
    verbose=True,
    n_jobs=2
)


In [ ]:
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score
)

acc1 = accuracy_score(y_train2, pred)
acc2 = accuracy_score(y_train2, pred2)
acc3 = accuracy_score(y_train2, pred3)
acc4 = accuracy_score(y_train2, pred4)

ps1 = precision_score(
    y_train2,
    pred,
    average="macro",
    zero_division=0
)
r1 = recall_score(
    y_train2,
    pred,
    average="macro",
    zero_division=0
)

ps2 = precision_score(
    y_train2,
    pred2,
    average="macro",
    zero_division=0
)
r2 = recall_score(
    y_train2,
    pred2,
    average="macro",
    zero_division=0
)

ps3 = precision_score(
    y_train2,
    pred3,
    average="macro",
    zero_division=0
)
r3 = recall_score(
    y_train2,
    pred3,
    average="macro",
    zero_division=0
)

ps4 = precision_score(
    y_train2,
    pred4,
    average="macro",
    zero_division=0
)
r4 = recall_score(
    y_train2,
    pred4,
    average="macro",
    zero_division=0
)

f1_geral = f1_score(
    y_train2,
    pred,
    average="macro"
)
f1_gera2 = f1_score(
    y_train2,
    pred2,
    average="macro"
)
f1_gera3 = f1_score(
    y_train2,
    pred3,
    average="macro"
)
f1_gera4 = f1_score(
    y_train2,
    pred4,
    average="macro"
)

df1_geral = f1_score(
    y_train2,
    pred,
    average=None
)
df1_gera2 = f1_score(
    y_train2,
    pred2,
    average=None
)
df1_gera3 = f1_score(
    y_train2,
    pred3,
    average=None
)
df1_gera4 = f1_score(
    y_train2,
    pred4,
    average=None
)


In [ ]:
print(
    "Perceptron - Accuracy:",
    acc1,
    ", F1_Score:",
    f1_geral,
    " Precision:",
    ps1,
    " Recall:",
    r1,
    "\n F1-Classe",
    df1_geral
)

print(
    "PassiveAggressiveClassifier - Accuracy:",
    acc2,
    ", F1_Score:",
    f1_gera2,
    " Precision:",
    ps2,
    " Recall:",
    r2,
    "\n F1-Classe",
    df1_gera2
)

print(
    "MultinomialNB - Accuracy:",
    acc3,
    ", F1_Score:",
    f1_gera3,
    " Precision:",
    ps3,
    " Recall:",
    r3,
    "\n F1-Classe",
    df1_gera3
)

print(
    "SGDClassifier - Accuracy:",
    acc4,
    ", F1_Score:",
    f1_gera4,
    " Precision:",
    ps4,
    " Recall:",
    r4,
    "\n F1-Classe",
    df1_gera4
)


In [ ]:
print(
    "Perceptron - Accuracy:",
    acc1,
    ", F1_Score:",
    f1_geral,
    "\n",
    df1_geral
)

print(
    "PassiveAggressiveClassifier - Accuracy:",
    acc2,
    ", F1_Score:",
    f1_gera2,
    "\n",
    df1_gera2
)

print(
    "MultinomialNB - Accuracy:",
    acc3,
    ", F1_Score:",
    f1_gera3,
    "\n",
    df1_gera3
)

print(
    "SGDClassifier - Accuracy:",
    acc4,
    ", F1_Score:",
    f1_gera4,
    "\n",
    df1_gera4
)


Resultado do feedback imediato


In [ ]:
perceptron = []
for i in range(1,15):
    Perceptron.treinar(i)
    perceptron.append(Perceptron.f1macro(i))
print(f"Perceptron: {perceptron}")


SGD = []
for i in range(1,15):
    SGDClassifier.treinar(i)
    SGD.append(SGDClassifier.f1macro(i))
print(f"SGDClassifier: {SGD}")

PassiveAggressiveClassifier_list = []
for i in range(1,15):
    PassiveAggressiveClassifier.treinar(i)
    PassiveAggressiveClassifier_list.append(PassiveAggressiveClassifier.f1macro(i))
print(f"PassiveAgressive: {PassiveAggressiveClassifier_list}")


MultinomialNB_list = []
for i in range(1,15):
    MultinomialNB.treinar(i)
    MultinomialNB_list.append(MultinomialNB.f1macro(i))
print(f"MultinomialNB: {MultinomialNB_list}")


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

n_meses = len(next(iter(resultados.values())))

if any(len(valores) != n_meses for valores in resultados.values()):
    raise ValueError("Todos os classificadores devem ter a mesma quantidade de resultados.")

meses = pd.date_range(
    start="2020-01-01",
    periods=n_meses,
    freq="MS"
).strftime("%b %Y")

estilos = {
    "Perceptron": ("orange", "o"),
    "SGDClassifier": ("firebrick", "s"),
    "PassiveAggressiveClassifier": ("blue", "^"),
    "MultinomialNB": ("green", "v"),
}

fig, ax = plt.subplots(figsize=(13, 8))

for classificador, valores in resultados.items():
    cor, marcador = estilos[classificador]

    ax.plot(
        meses,
        valores,
        color=cor,
        marker=marcador,
        linewidth=1.4,
        markersize=7,
        label=classificador
    )

ax.set_ylabel("F-measure", fontsize=14)
ax.set_ylim(0.35, 1.01)
ax.set_yticks(np.arange(0.36, 1.01, 0.02))
ax.grid(axis="y", color="gray", alpha=0.5)
ax.tick_params(axis="x", rotation=50)

ax.legend(
    loc="lower left",
    bbox_to_anchor=(0, 1.01, 1, 0.2),
    mode="expand",
    ncol=4
)

plt.tight_layout()
plt.savefig(str(RESULT_DIR / "resultados_feedback.png"), dpi=500, bbox_inches="tight")
plt.show()